### DESCRIPTION (inspired by Leetcode.com)
medium

A distributed system has n servers labeled 1 to n. The servers communicate through directed connections, where each connection [from, to, latency] means server from can send a message to server to with the given latency (in milliseconds).

When server k broadcasts an alert, it propagates through the network along all available paths. Return the minimum time required for every server to receive the alert. If some servers are unreachable from k, return -1.

Example 1:

Input:

connections = [[1,2,3], [1,3,5], [2,3,1]]
n = 3
k = 1

Output: 4

Explanation: There are two paths to server 3: the direct path 1→3 takes 5ms, but the path 1→2→3 takes only 3+1=4ms. The last server to receive the alert determines the answer.

Example 2:

Input:

connections = [[1,2,2], [3,2,1]]
n = 3
k = 1

Output:-1

Explanation: The edge [3,2,1] points from server 3 to server 2, not the other way around. Server 1 can reach server 2, but there's no path from server 1 to server 3.

In [ ]:
from typing import List
import heapq

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        # 1. convert times list to graph, so that graph[node] = [(neighbor1, delay), (neighbor2, delay) ...]
        graph = [[] for _ in range(n+1)] 
        for src, dest, delay in times:
            graph[src].append((dest, delay))

        # 2. create dictionary to track the shortest time to each node from source (k), starting from k itself, shortest[k] = 0
        # put K and current time to it into heap 
        shortest = {k:0}
        heap = [(0,k)] # must have time before the node in the tuple, since the heap is sort by time

        # 3. while heapq is not empty, pop each node and time-to-node. If time-to-node is greater than the one we have in dictionary, skip this iteration.
        # Else process node by going through each of its neighbor. Add time to neighbor and time to node to calculate total time from source. 
        # If this new time is less than what we have in dictionary, update the value in dictionary.
        while heap:
            time_to_node, node = heapq.heappop(heap)
            if time_to_node > shortest.get(node, float('inf')):
                continue
                
            for neighbor, time_to_neighbor in graph[node]:
                new_time = time_to_node + time_to_neighbor
                if new_time < shortest.get(neighbor, float('inf')):
                    shortest[neighbor] = new_time
                    heapq.heappush(heap, (new_time, neighbor))

        # 4. check if dictionary have same length as n. If not, some nodes are not reachable return -1. Otherwise, return the largest value in the dictionary.  
        return max(shortest.values()) if len(shortest) == n else -1

### Feedback

Correct and efficient Dijkstra solution—congratulations. Your adjacency-list construction, stale-heap-entry check, relaxation logic, and reachability test are all sound. It runs in O((n + E) log n) time and O(n + E) space. 

A couple of interview-ready refinements: use consistent four-space indentation (the method currently has an extra leading space before comments/code), and add the type import if the execution environment does not predefine List (from typing import List). You could also early-exit when all n nodes have been finalized, but that is optional and not needed for correctness.


In [4]:
from typing import Callable

class Input:
    def __init__(self, times: List[List[int]], n: int, k: int):
        self.times = times
        self.n = n
        self.k = k
        
class Test:  
    def __init__(self, input: Input, result: bool):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[List[List[int]], int, int], int]):
    for test in tests:
        result = func(test.input.times, test.input.n, test.input.k)
        if result == test.expected_result:
            print(f"Test passed for {test.input.times}")
        else:
            print(f"Test failed for {test.input.times}. Expected: {test.expected_result}, Actual: {result}")

In [19]:
tests = [
    Test(Input([[1,2,2]], n=2, k=2), -1),
    Test(Input([[1,2,2], [2,1,1]], n=2, k=1), 2),
    Test(Input([[1,2,2], [2,1,1]], n=2, k=2), 1),
    Test(Input([[1,2,3], [1,3,5], [2,3,1]], n=3, k=1), 4),
    Test(Input([[1,2,2], [3,2,1]], n=3, k=1), -1),
]

run_tests(tests, Solution().networkDelayTime)

Test passed for [[1, 2, 2]]
Test passed for [[1, 2, 2], [2, 1, 1]]
Test passed for [[1, 2, 2], [2, 1, 1]]
Test passed for [[1, 2, 3], [1, 3, 5], [2, 3, 1]]
Test passed for [[1, 2, 2], [3, 2, 1]]
